# ARC2 011 CPU preflight

Manual bounded research experiment, 2026-10-11. Fixed Qwen3.8-27B and verified S4 step25; no competition submission. Results below are produced by execution, not assumed. CPU preflight does not load model weights. GPU mode has a 120-minute notebook budget and a 20-minute model-load stop; it does not retry.


### 1. Restore frozen code and input-only task packets


In [ ]:
import base64, hashlib, io, json, pathlib, time, zipfile
NOTEBOOK_START=time.monotonic()
CODE=pathlib.Path('/kaggle/working/011/code'); CODE.mkdir(parents=True,exist_ok=True)
CODE_MANIFEST={'arc_episode_data.py': '7c672182a1bff3654231f9e60da2ba503ae94124b215d04857cb20345d93c864', 'grid_runtime.py': 'aa3d41021998e0dc9f47b532a63ad86ba6baaeec5991a1179ccef8570a6eb4e0', 'p12-challenges.json': '792ca71c069abb420368efd1b0a29c04adaaed9761ba9225528db735cfe44a40', 'p12-queries.json': '884eb10df5a65bb527cd4c4c72edc7b5ce4dafd16f481e8fef8cc8478e7950f7', 'p12-scoring.json': 'fbe21ed91e38259e70aec5864994944759e2ec83dd0bf2c9093a7c549c562459', 'p6-queries.json': '02e7a837fddd9a46d4b8af2e0ef630acafd55655cfce10af73633be8134e2844', 'research-manifest.json': '301c10ad8768c232ba50a2b94cc9b3b4c975da9fdc3ea936b90a02719575ffeb', 'run_budgeted_grid_experiment.py': '4d4ab129889af0ebf3a89bc87f83691d71014db3d6c118fafa6b3f6499c2f9ad', 'solve_grid.py': 'c29c428d87873c32be29fd394b8d06cdcee37da694723d49a677d25f699ca190', 'solve_program.py': 'bb500aef69a90b9613f74e0e33b44895f7d3b9771138481b500ba836605bb54d', 'train_grid_sft.py': '2e4ccb40b7acee2c3b5659eff9a6afa4bcd57fa65ee78f5a1ee6e1dda285f044', 'ttt_per_task.py': '17d32077cc788457ae6943de6c4944453c280d4ef515220915920321a42d7229'}
EXPECTED_ADAPTER_SHA='9282c46cbce02923afbea3ad041614f0bc6ff3d7bd97c5174adfa3dd3e485ff9'
payload=base64.b64decode('...')
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    for name in z.namelist():
        if pathlib.PurePosixPath(name).name != name: raise ValueError('unsafe archive path')
        data=z.read(name)
        assert hashlib.sha256(data).hexdigest()==CODE_MANIFEST[name]
        (CODE/name).write_bytes(data)
(CODE/'source-manifest.json').write_text(json.dumps(CODE_MANIFEST,indent=2))
print('PACKAGED_INPUTS',len(CODE_MANIFEST))


### 2. Prepare the existing offline environment


In [ ]:
import glob, os, subprocess, sys, time, json
wheel = glob.glob("/kaggle/input/**/qwen38-wheelhouse-py312", recursive=True)
if len(wheel) != 1:
    raise RuntimeError("Expected one frozen wheelhouse, found %d" % len(wheel))
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
               stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT, timeout=90)
p = subprocess.run([sys.executable, "-m", "pip", "install", "--no-index",
                    "--find-links=" + wheel[0], "transformers==5.8.0", "accelerate", "peft"],
                   capture_output=True, text=True, timeout=240)
print((p.stdout or "")[-1400:])
if p.returncode:
    print((p.stderr or "")[-1400:])
    raise RuntimeError("Offline environment preparation failed")
import transformers, torch, peft, accelerate
versions = dict(python=sys.version.split()[0], transformers=transformers.__version__,
                torch=torch.__version__, peft=peft.__version__, accelerate=accelerate.__version__)
print("VERSIONS", json.dumps(versions))
assert transformers.__version__ == "5.8.0"


### 3. Verify model and adapter identities


In [ ]:
from pathlib import Path
import hashlib
def digest(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(2*1024*1024), b""):
            h.update(b)
    return h.hexdigest()
models = []
for p in Path("/kaggle/input").rglob("config.json"):
    try:
        c = json.loads(p.read_text())
    except Exception:
        continue
    if c.get("model_type") == "qwen3_5" and (p.parent/"model.safetensors.index.json").is_file():
        models.append(p.parent)
if len(models) != 1:
    raise RuntimeError("Expected one Qwen3.8 model directory, found %d" % len(models))
model_dir = str(models[0])
adapters = list(Path("/kaggle/input").rglob("adapter_step_25/adapter_model.safetensors"))
matching = [p.parent for p in adapters if digest(p) == EXPECTED_ADAPTER_SHA]
if len(matching) != 1:
    raise RuntimeError("Expected verified S4 adapter SHA, matching directories=%d" % len(matching))
adapter_dir = str(matching[0])
assert json.loads((matching[0]/"formatter.json").read_text())["formatter"] == "f0_space"
print("INPUT_IDENTITY", json.dumps(dict(model_dir=model_dir, adapter_dir=adapter_dir,
     adapter_sha256=EXPECTED_ADAPTER_SHA, model_config_sha256=digest(models[0]/"config.json"))))
sys.path.insert(0, str(CODE))


### 4. Freeze the two development stages

P12 F0-25 closes the previous comparison. P6 base F1 original/sample/transpose is a predeclared diagnostic. No confirmation tasks are loaded. Labels are available only to the post-generation scorer.


In [ ]:
config = {
 "model_dir": model_dir, "challenges": str(CODE/"p12-challenges.json"),
 "solutions": str(CODE/"p12-scoring.json"), "out_dir": "/kaggle/working/011/s0-shared",
 "time_budget_s": 7200, "load_limit_s": 1200,
 "notebook_start_monotonic": NOTEBOOK_START,
 "runtime": {"base_source":"kaggle:ugvfpdcuwfnh/qwen3-8-27b/Transformers/transformers-bf16/1",
             "deactivate_async_load":True, "save_window_s":300},
 "stages": [
  {"type":"grid","name":"s0_f0_25_p12","adapter":adapter_dir,"formatter":"f0_space",
   "query_file":str(CODE/"p12-queries.json"),"max_new_tokens":2048,
   "max_time_s":600,"estimate_s":240,
   "candidates":[{"view":"original","do_sample":False}]},
  {"type":"grid","name":"s1_base_f1_p6","adapter":None,"formatter":"f1_compact",
   "query_file":str(CODE/"p6-queries.json"),"max_new_tokens":960,
   "time_budget_s":1200,"max_time_s":300,"estimate_s":120,
   "candidates":[{"view":"original","do_sample":False},
                 {"view":"original","do_sample":True,"sample_index":1},
                 {"view":"transpose","do_sample":False}]}
 ]}
config_path=CODE/"run.json"
config_path.write_text(json.dumps(config,ensure_ascii=False,indent=2))
p=subprocess.run([sys.executable,str(CODE/"run_budgeted_grid_experiment.py"),
                  "--config",str(config_path),"--dry-run"],capture_output=True,text=True,timeout=60)
print(p.stdout)
if p.returncode:
    print(p.stderr[-1500:])
    raise RuntimeError("Configuration preflight failed")


### 5. CPU diagnostics


In [ ]:
# Small manually run CPU diagnostics; no model weights or CUDA session are loaded.
from transformers import AutoTokenizer
from arc_episode_data import build_messages
import solve_program, ttt_per_task
from grid_runtime import CandidateJournal, canonical_hash
tok=AutoTokenizer.from_pretrained(model_dir,local_files_only=True,trust_remote_code=False)
tasks=json.loads((CODE/"p12-challenges.json").read_text())
sample=tasks["db118e2a"]
msg=build_messages(sample["train"],sample["test"][0]["input"],"f1_compact")
off=tok.apply_chat_template(msg,tokenize=False,add_generation_prompt=True,enable_thinking=False)
on=tok.apply_chat_template(msg,tokenize=False,add_generation_prompt=True,
                           enable_thinking=True,reasoning_effort="low",preserve_thinking=True)
assert off != on and on.endswith("<think>\n")
executor=solve_program.executor_preflight()
assert executor["status"]=="ok",executor
cases=[
 ("identity","def transform(grid):\n    return [row[:] for row in grid]",2.0,"ok"),
 ("exception","def transform(grid):\n    return grid[100]",2.0,"execution_error"),
 ("timeout","def transform(grid):\n    while True:\n        pass",0.2,"execution_timeout"),
 ("import","import os\ndef transform(grid):\n    return grid",2.0,"program_rejected"),
]
case_results={}
for name,code,wall,want in cases:
    result=solve_program.execute_grid(code,[[0,1],[2,3]],wall_seconds=wall,memory_mb=512)
    case_results[name]=result
    assert result["status"]==want,(name,result)
assert case_results["identity"]["grid"]==[[0,1],[2,3]]
assert not torch.cuda.is_available(),"CPU diagnostic must not reserve GPU"
report={"status":"CPU_PREFLIGHT_OK","model_weights_loaded":False,"gpu_count":torch.cuda.device_count(),
        "versions":versions,"code_manifest":CODE_MANIFEST,"adapter_sha256":EXPECTED_ADAPTER_SHA,
        "executor":executor,"case_results":case_results,
        "thinking_prompt_tokens":len(tok.encode(on,add_special_tokens=False)),
        "nonthinking_prompt_tokens":len(tok.encode(off,add_special_tokens=False)),
        "elapsed_s":time.monotonic()-NOTEBOOK_START}
Path("/kaggle/working/011/cpu-preflight.json").write_text(json.dumps(report,ensure_ascii=False,indent=2))
print(json.dumps(report,ensure_ascii=False,indent=2))


### Interpretation

Read the generated JSON and worker journal. A successful CPU diagnostic establishes preparation only. GPU outcomes must include all requested queries, explicit incomplete cases, and actual timing. No result here is a competition score.
